# CSQA Layerwise Decoder Analysis With Plain Transformers

This notebook analyzes CommonsenseQA using a decoder-only model directly through 	ransformers, without the repository's trace extraction pipeline.

The decision space is the task-relevant constrained answer set Aâ€“E, not the full vocabulary. For this task that is the right object to analyze, because evaluation is defined by which answer option the model selects at the Answer: position.

Default model choice: Qwen/Qwen2.5-3B-Instruct.

Why this model:

- it is a decoder-only causal LM with a standard qwen2 architecture
- it is a pragmatic choice for this notebook because it is strong enough for CSQA-style reasoning and loads cleanly in mainstream 	ransformers builds


In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data.load_csqa import load_csqa

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


## Configuration

LIMIT=None runs the whole split. MAX_SEQ_LEN truncates from the left, matching the causal answer-slot setup.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SPLIT = "validation"
LIMIT = None
MAX_SEQ_LEN = 256
BATCH_SIZE = 4
CV_SPLITS = 5
SEED = 42
USE_BFLOAT16_IF_AVAILABLE = True
VOCAB_ENTROPY_SATURATION_THRESHOLD = 0.20
TRUE_ANSWER_MARGIN_OVER_VOCAB_THRESHOLD = 0.0

## Data And Model Setup


In [3]:
LETTERS = ["A", "B", "C", "D", "E"]

rows = load_csqa(split=SPLIT, limit=LIMIT).copy()
rows["n_choices"] = rows["csqa_choices"].map(len)
rows["prompt_len_chars"] = rows["text"].str.len()

assert rows["n_choices"].eq(5).all(), "Expected 5 choices for every CSQA row."

if torch.cuda.is_available():
    if USE_BFLOAT16_IF_AVAILABLE and torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    else:
        model_dtype = torch.float16
    device_map = "auto"
else:
    model_dtype = torch.float32
    device_map = None

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map=device_map,
    attn_implementation="eager",
)
model.eval()

def build_answer_token_ids(tok):
    out = {}
    for letter in LETTERS:
        ids = tok(" " + letter, add_special_tokens=False)["input_ids"]
        if len(ids) != 1:
            raise ValueError(f"Answer token '{letter}' is not single-token: {ids}")
        out[letter] = int(ids[0])
    return out


answer_token_ids = build_answer_token_ids(tok)
answer_ids = [answer_token_ids[l] for l in LETTERS]
answer_id_tensor = torch.tensor(answer_ids, dtype=torch.long)

display(rows[["example_id", "answerKey", "prompt_len_chars"]].head())
print("rows:", len(rows))
print("answer token ids:", answer_token_ids)


Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

,example_id,answerKey,prompt_len_chars
0,701fac8b8c04ab56c4394b2e7b2aa8df,A,187
1,1db37ef1b4ebdcbc12a7d7dec87472a8,A,145
2,a5cb28d53c4cd35c06fbbbb3dbf6aaa8,B,149
3,ce6c00c4f3edcd54ab4fa1e3a8582e7c,A,141
4,054aade2878f1bd189aa590565583d91,A,158


rows: 1221
answer token ids: {'A': 362, 'B': 425, 'C': 356, 'D': 422, 'E': 468}


## Layerwise Constrained Extraction

For each prompt, the notebook runs a single forward pass with output_hidden_states=True, takes the hidden state at the final prompt position, projects every layer onto the five answer tokens, and computes constrained decision statistics over depth.


In [ ]:
def get_final_norm_module(model):
    candidates = [
        "model.norm",
        "model.final_layernorm",
        "transformer.ln_f",
        "gpt_neox.final_layer_norm",
    ]
    for path in candidates:
        cur = model
        ok = True
        for part in path.split("."):
            if not hasattr(cur, part):
                ok = False
                break
            cur = getattr(cur, part)
        if ok:
            return cur
    return None


def encode_prompts(texts, tok, max_seq_len):
    batch = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
        padding=True,
        return_tensors="pt",
    )
    pos = []
    for mask in batch["attention_mask"]:
        nz = torch.nonzero(mask, as_tuple=False).view(-1)
        pos.append(int(nz[-1].item()))
    batch["decision_pos"] = torch.tensor(pos, dtype=torch.long)
    return batch


def softmax_rows(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def entropy_from_probabilities(probs):
    probs = np.clip(probs, 1e-12, None)
    return -(probs * np.log(probs)).sum(axis=1)


def first_true(mask):
    idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    return float(idx[0]) if len(idx) else np.nan


def first_stable(mask):
    mask = np.asarray(mask, dtype=bool)
    for i in range(len(mask)):
        if mask[i] and mask[i:].all():
            return float(i)
    return np.nan


def first_numeric_stable(values):
    values = np.asarray(values)
    for i in range(len(values)):
        if np.all(values[i:] == values[i]):
            return float(i)
    return np.nan


def first_threshold_crossing(values, threshold, direction="le"):
    values = np.asarray(values, dtype=float)
    if direction == "le":
        idx = np.flatnonzero(values <= threshold)
    elif direction == "ge":
        idx = np.flatnonzero(values >= threshold)
    else:
        raise ValueError(direction)
    return float(idx[0]) if len(idx) else np.nan


def first_threshold_stable(values, threshold, direction="le"):
    values = np.asarray(values, dtype=float)
    for i in range(len(values)):
        tail = values[i:]
        if direction == "le" and np.all(tail <= threshold):
            return float(i)
        if direction == "ge" and np.all(tail >= threshold):
            return float(i)
    return np.nan


def rank_bucket(rank):
    rank = int(rank)
    if rank == 1:
        return "rank1_correct"
    if rank == 2:
        return "rank2_near_miss"
    return "rank3plus_hard"


final_norm = get_final_norm_module(model)
lm_head_weight = model.lm_head.weight.detach()
vocab_size = int(lm_head_weight.shape[0])
log_vocab_size = float(np.log(vocab_size))
log_answer_choice_count = float(np.log(len(LETTERS)))

probe_cpu = encode_prompts(rows["text"].head(1), tok, MAX_SEQ_LEN)
probe_pos = int(probe_cpu["decision_pos"][0].item())
probe = {k: v.to(model.device) for k, v in probe_cpu.items() if k != "decision_pos"}
with torch.no_grad():
    probe_out = model(**probe, output_hidden_states=True, return_dict=True, use_cache=False)

hidden_probe = probe_out.hidden_states
L_plus_1 = len(hidden_probe)
L = L_plus_1 - 1

choice_w = lm_head_weight[answer_id_tensor.to(lm_head_weight.device)].float()
target_logits = probe_out.logits[0, probe_pos, answer_id_tensor.to(probe_out.logits.device)].float().detach().cpu()
raw_last = hidden_probe[-1][0, probe_pos].float()
raw_logits = torch.mv(choice_w.cpu(), raw_last.detach().cpu())

if final_norm is not None:
    normed_last = final_norm(raw_last.unsqueeze(0)).squeeze(0)
    normed_logits = torch.mv(choice_w.cpu(), normed_last.detach().cpu())
    raw_err = torch.mean(torch.abs(raw_logits - target_logits)).item()
    normed_err = torch.mean(torch.abs(normed_logits - target_logits)).item()
    LAST_LAYER_NEEDS_NORM = bool(normed_err < raw_err)
else:
    LAST_LAYER_NEEDS_NORM = False


def maybe_apply_final_norm(h, layer_idx):
    if final_norm is None:
        return h
    if layer_idx < L:
        return final_norm(h)
    if LAST_LAYER_NEEDS_NORM:
        return final_norm(h)
    return h


analysis_rows = []

for start in tqdm(range(0, len(rows), BATCH_SIZE), total=int(np.ceil(len(rows) / BATCH_SIZE)), desc="csqa hidden-state pass"):
    batch_df = rows.iloc[start:start + BATCH_SIZE].reset_index(drop=True)
    batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
    decision_pos = batch_cpu.pop("decision_pos")
    batch = {k: v.to(model.device) for k, v in batch_cpu.items()}
    decision_pos = decision_pos.to(model.device)

    with torch.no_grad():
        out = model(**batch, output_hidden_states=True, return_dict=True, use_cache=False)

    hidden_states = out.hidden_states
    answer_choice_weight = lm_head_weight[answer_id_tensor.to(lm_head_weight.device)].float()
    batch_answer_choice_logits_per_layer = []
    batch_full_vocab_predicted_token_id_per_layer = []
    batch_vocab_entropy_per_layer = []
    batch_vocab_entropy_normalized_per_layer = []
    batch_answer_choice_entropy_per_layer = []
    batch_answer_choice_entropy_normalized_per_layer = []
    batch_true_answer_probability_over_vocab_per_layer = []
    batch_true_answer_rank_over_vocab_per_layer = []
    batch_true_answer_margin_over_vocab_per_layer = []
    true_answer_token_ids = answer_id_tensor.to(batch["input_ids"].device)[
        torch.tensor(
            [LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()],
            device=batch["input_ids"].device,
            dtype=torch.long,
        )
    ]

    for li in range(L_plus_1):
        h = hidden_states[li][torch.arange(hidden_states[li].shape[0], device=decision_pos.device), decision_pos].float()
        h = maybe_apply_final_norm(h, li)
        full_vocab_logits = torch.matmul(h, lm_head_weight.T.float())
        full_vocab_log_probs = torch.log_softmax(full_vocab_logits, dim=-1)
        full_vocab_probs = torch.exp(full_vocab_log_probs)
        full_vocab_entropy = -(full_vocab_probs * full_vocab_log_probs).sum(dim=-1)
        true_answer_logit_over_vocab = full_vocab_logits.gather(1, true_answer_token_ids.unsqueeze(1)).squeeze(1)
        true_answer_probability_over_vocab = full_vocab_probs.gather(1, true_answer_token_ids.unsqueeze(1)).squeeze(1)
        true_answer_rank_over_vocab = 1 + (full_vocab_logits > true_answer_logit_over_vocab.unsqueeze(1)).sum(dim=-1)
        top2_values, top2_indices = torch.topk(full_vocab_logits, k=2, dim=-1)
        best_other_logit_over_vocab = torch.where(
            top2_indices[:, 0].eq(true_answer_token_ids),
            top2_values[:, 1],
            top2_values[:, 0],
        )
        true_answer_margin_over_vocab = true_answer_logit_over_vocab - best_other_logit_over_vocab
        full_vocab_predicted_token_id = torch.argmax(full_vocab_logits, dim=-1)

        answer_choice_logits = full_vocab_logits.index_select(-1, answer_id_tensor.to(full_vocab_logits.device))
        answer_choice_log_probs = torch.log_softmax(answer_choice_logits, dim=-1)
        answer_choice_probs = torch.exp(answer_choice_log_probs)
        answer_choice_entropy = -(answer_choice_probs * answer_choice_log_probs).sum(dim=-1)

        batch_answer_choice_logits_per_layer.append(answer_choice_logits.detach().cpu().numpy())
        batch_full_vocab_predicted_token_id_per_layer.append(full_vocab_predicted_token_id.detach().cpu().numpy())
        batch_vocab_entropy_per_layer.append(full_vocab_entropy.detach().cpu().numpy())
        batch_vocab_entropy_normalized_per_layer.append((full_vocab_entropy / log_vocab_size).detach().cpu().numpy())
        batch_answer_choice_entropy_per_layer.append(answer_choice_entropy.detach().cpu().numpy())
        batch_answer_choice_entropy_normalized_per_layer.append((answer_choice_entropy / log_answer_choice_count).detach().cpu().numpy())
        batch_true_answer_probability_over_vocab_per_layer.append(true_answer_probability_over_vocab.detach().cpu().numpy())
        batch_true_answer_rank_over_vocab_per_layer.append(true_answer_rank_over_vocab.detach().cpu().numpy())
        batch_true_answer_margin_over_vocab_per_layer.append(true_answer_margin_over_vocab.detach().cpu().numpy())

    batch_answer_choice_logits = np.stack(batch_answer_choice_logits_per_layer, axis=1)
    batch_full_vocab_predicted_token_id_curve = np.stack(batch_full_vocab_predicted_token_id_per_layer, axis=1)
    batch_vocab_entropy_curve = np.stack(batch_vocab_entropy_per_layer, axis=1)
    batch_vocab_entropy_normalized_curve = np.stack(batch_vocab_entropy_normalized_per_layer, axis=1)
    batch_answer_choice_entropy_curve = np.stack(batch_answer_choice_entropy_per_layer, axis=1)
    batch_answer_choice_entropy_normalized_curve = np.stack(batch_answer_choice_entropy_normalized_per_layer, axis=1)
    batch_true_answer_probability_over_vocab_curve = np.stack(batch_true_answer_probability_over_vocab_per_layer, axis=1)
    batch_true_answer_rank_over_vocab_curve = np.stack(batch_true_answer_rank_over_vocab_per_layer, axis=1)
    batch_true_answer_margin_over_vocab_curve = np.stack(batch_true_answer_margin_over_vocab_per_layer, axis=1)

    for bi in range(batch_answer_choice_logits.shape[0]):
        answer_choice_logits = batch_answer_choice_logits[bi]
        full_vocab_predicted_token_id_curve = batch_full_vocab_predicted_token_id_curve[bi]
        answer_choice_probabilities = softmax_rows(answer_choice_logits)
        answer_choice_top1_top2_logit_gap_curve = np.sort(answer_choice_logits, axis=1)[:, ::-1][:, 0] - np.sort(answer_choice_logits, axis=1)[:, ::-1][:, 1]
        predicted_answer_choice_index_curve = np.argsort(answer_choice_logits, axis=1)[:, ::-1][:, 0]
        vocab_entropy_curve = batch_vocab_entropy_curve[bi]
        vocab_entropy_normalized_curve = batch_vocab_entropy_normalized_curve[bi]
        answer_choice_entropy_curve = batch_answer_choice_entropy_curve[bi]
        answer_choice_entropy_normalized_curve = batch_answer_choice_entropy_normalized_curve[bi]
        entropy_normalized_difference_curve = vocab_entropy_normalized_curve - answer_choice_entropy_normalized_curve
        true_answer_probability_over_vocab_curve = batch_true_answer_probability_over_vocab_curve[bi]
        true_answer_rank_over_vocab_curve = batch_true_answer_rank_over_vocab_curve[bi]
        true_answer_margin_over_vocab_curve = batch_true_answer_margin_over_vocab_curve[bi]

        true_letter = str(batch_df.loc[bi, "answerKey"])
        true_idx = LETTERS.index(true_letter)
        true_answer_probability_within_choices_curve = answer_choice_probabilities[:, true_idx]
        true_answer_rank_within_choices_curve = 1 + (answer_choice_logits > answer_choice_logits[:, [true_idx]]).sum(axis=1)
        true_answer_margin_within_choices_curve = answer_choice_logits[:, true_idx] - np.max(
            np.where(np.arange(answer_choice_logits.shape[1])[None, :] == true_idx, -np.inf, answer_choice_logits),
            axis=1,
        )

        final_pred_idx = int(predicted_answer_choice_index_curve[-1])
        final_pred_letter = LETTERS[final_pred_idx]
        final_rank = int(true_answer_rank_within_choices_curve[-1])
        final_full_vocab_prediction_token_id = int(full_vocab_predicted_token_id_curve[-1])
        full_vocab_prediction_agreement_with_final_curve = (full_vocab_predicted_token_id_curve == final_full_vocab_prediction_token_id)
        answer_choice_prediction_agreement_with_final_curve = (predicted_answer_choice_index_curve == final_pred_idx)
        first_low_vocab_entropy_layer = first_threshold_crossing(
            vocab_entropy_normalized_curve,
            VOCAB_ENTROPY_SATURATION_THRESHOLD,
            direction="le",
        )
        low_vocab_entropy_stable_layer = first_threshold_stable(
            vocab_entropy_normalized_curve,
            VOCAB_ENTROPY_SATURATION_THRESHOLD,
            direction="le",
        )
        true_answer_rank_stable_layer = first_numeric_stable(true_answer_rank_over_vocab_curve)
        true_answer_margin_over_vocab_stable_layer = first_threshold_stable(
            true_answer_margin_over_vocab_curve,
            TRUE_ANSWER_MARGIN_OVER_VOCAB_THRESHOLD,
            direction="ge",
        )

        analysis_rows.append(
            {
                "example_id": batch_df.loc[bi, "example_id"],
                "answerKey": true_letter,
                "final_answer_choice_prediction": final_pred_letter,
                "final_answer_choice_is_correct": bool(final_pred_letter == true_letter),
                "final_answer_choice_true_rank": final_rank,
                "answer_choice_rank_bucket": rank_bucket(final_rank),
                "answer_choice_hit_at_2": bool(final_rank <= 2),
                "answer_choice_hit_at_3": bool(final_rank <= 3),
                "final_vocab_entropy": float(vocab_entropy_curve[-1]),
                "final_vocab_entropy_normalized": float(vocab_entropy_normalized_curve[-1]),
                "final_answer_choice_entropy": float(answer_choice_entropy_curve[-1]),
                "final_answer_choice_entropy_normalized": float(answer_choice_entropy_normalized_curve[-1]),
                "final_entropy_normalized_difference": float(entropy_normalized_difference_curve[-1]),
                "final_true_answer_probability_over_vocab": float(true_answer_probability_over_vocab_curve[-1]),
                "final_true_answer_rank_over_vocab": float(true_answer_rank_over_vocab_curve[-1]),
                "final_true_answer_margin_over_vocab": float(true_answer_margin_over_vocab_curve[-1]),
                "final_answer_choice_top1_top2_logit_gap": float(answer_choice_top1_top2_logit_gap_curve[-1]),
                "final_true_answer_probability_within_choices": float(true_answer_probability_within_choices_curve[-1]),
                "final_true_answer_margin_within_choices": float(true_answer_margin_within_choices_curve[-1]),
                "answer_choice_winner_flip_count": int(np.sum(predicted_answer_choice_index_curve[1:] != predicted_answer_choice_index_curve[:-1])),
                "final_answer_choice_winner_first_layer": first_true(answer_choice_prediction_agreement_with_final_curve),
                "final_answer_choice_winner_stable_layer": first_stable(answer_choice_prediction_agreement_with_final_curve),
                "first_low_vocab_entropy_layer": first_low_vocab_entropy_layer,
                "low_vocab_entropy_stable_layer": low_vocab_entropy_stable_layer,
                "true_answer_rank_stable_layer": true_answer_rank_stable_layer,
                "true_answer_margin_over_vocab_stable_layer": true_answer_margin_over_vocab_stable_layer,
                "first_full_vocab_agreement_with_final_layer": first_true(full_vocab_prediction_agreement_with_final_curve),
                "stable_full_vocab_agreement_with_final_layer": first_stable(full_vocab_prediction_agreement_with_final_curve),
                "first_answer_choice_agreement_with_final_layer": first_true(answer_choice_prediction_agreement_with_final_curve),
                "stable_answer_choice_agreement_with_final_layer": first_stable(answer_choice_prediction_agreement_with_final_curve),
                "vocab_entropy_curve": vocab_entropy_curve.astype(np.float32),
                "vocab_entropy_normalized_curve": vocab_entropy_normalized_curve.astype(np.float32),
                "answer_choice_entropy_curve": answer_choice_entropy_curve.astype(np.float32),
                "answer_choice_entropy_normalized_curve": answer_choice_entropy_normalized_curve.astype(np.float32),
                "entropy_normalized_difference_curve": entropy_normalized_difference_curve.astype(np.float32),
                "true_answer_probability_over_vocab_curve": true_answer_probability_over_vocab_curve.astype(np.float32),
                "true_answer_rank_over_vocab_curve": true_answer_rank_over_vocab_curve.astype(np.float32),
                "true_answer_margin_over_vocab_curve": true_answer_margin_over_vocab_curve.astype(np.float32),
                "answer_choice_top1_top2_logit_gap_curve": answer_choice_top1_top2_logit_gap_curve.astype(np.float32),
                "true_answer_probability_within_choices_curve": true_answer_probability_within_choices_curve.astype(np.float32),
                "true_answer_rank_within_choices_curve": true_answer_rank_within_choices_curve.astype(np.float32),
                "true_answer_margin_within_choices_curve": true_answer_margin_within_choices_curve.astype(np.float32),
                "full_vocab_prediction_agreement_with_final_curve": full_vocab_prediction_agreement_with_final_curve.astype(np.float32),
                "answer_choice_prediction_agreement_with_final_curve": answer_choice_prediction_agreement_with_final_curve.astype(np.float32),
            }
        )


analysis_df = pd.DataFrame(analysis_rows)
bucket_order = ["rank1_correct", "rank2_near_miss", "rank3plus_hard"]
bucket_colors = {
    "rank1_correct": "#2a9d8f",
    "rank2_near_miss": "#e9c46a",
    "rank3plus_hard": "#e76f51",
}

display(
    pd.DataFrame(
        {
            "count": analysis_df["answer_choice_rank_bucket"].value_counts().reindex(bucket_order),
            "share": analysis_df["answer_choice_rank_bucket"].value_counts(normalize=True).reindex(bucket_order),
        }
    ).round(4)
)

print("L+1:", L_plus_1)
print("last layer needs final norm:", LAST_LAYER_NEEDS_NORM)
print("constrained top-1 accuracy:", round(float(analysis_df["final_answer_choice_is_correct"].mean()), 4))
print("constrained hit@2:", round(float(analysis_df["answer_choice_hit_at_2"].mean()), 4))
print("constrained hit@3:", round(float(analysis_df["answer_choice_hit_at_3"].mean()), 4))


csqa hidden-state pass:   0%|          | 0/1221 [00:00<?, ?it/s]

## Final-Layer And Layerwise Analysis

The key entropy objects in this notebook are:

- ocab_entropy_normalized_curve: entropy over the full vocabulary at each layer, normalized by log(vocab_size)
- nswer_choice_entropy_normalized_curve: entropy over the constrained A-E answer distribution at each layer, normalized by log(5)

This makes the two entropy curves directly comparable across layers and across examples.


In [ ]:
summary_cols = [
    "final_vocab_entropy_normalized",
    "final_answer_choice_entropy_normalized",
    "final_entropy_normalized_difference",
    "final_true_answer_probability_over_vocab",
    "final_true_answer_rank_over_vocab",
    "final_true_answer_margin_over_vocab",
    "first_low_vocab_entropy_layer",
    "true_answer_rank_stable_layer",
    "true_answer_margin_over_vocab_stable_layer",
    "final_answer_choice_top1_top2_logit_gap",
    "final_true_answer_probability_within_choices",
    "final_true_answer_margin_within_choices",
    "answer_choice_winner_flip_count",
    "final_answer_choice_winner_first_layer",
    "final_answer_choice_winner_stable_layer",
]

display(
    analysis_df.groupby("answer_choice_rank_bucket")[summary_cols]
    .agg(["mean", "median", "std"])
    .round(4)
)


In [ ]:
plot_cols = [
    "final_vocab_entropy_normalized",
    "final_answer_choice_entropy_normalized",
    "final_entropy_normalized_difference",
    "final_true_answer_margin_over_vocab",
    "final_answer_choice_top1_top2_logit_gap",
    "answer_choice_winner_flip_count",
]

titles = {
    "final_vocab_entropy_normalized": "Final normalized full-vocabulary entropy",
    "final_answer_choice_entropy_normalized": "Final normalized answer-choice entropy",
    "final_entropy_normalized_difference": "Final normalized entropy difference",
    "final_true_answer_margin_over_vocab": "Final true-answer margin over full vocabulary",
    "final_answer_choice_top1_top2_logit_gap": "Final answer-choice top1-top2 logit gap",
    "answer_choice_winner_flip_count": "Answer-choice winner flips over layers",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, plot_cols):
    for bucket in bucket_order:
        vals = pd.to_numeric(
            analysis_df.loc[analysis_df["answer_choice_rank_bucket"].eq(bucket), col],
            errors="coerce",
        ).dropna()
        ax.hist(vals, bins=35, density=True, alpha=0.35, color=bucket_colors[bucket], label=bucket)
    ax.set_title(titles[col])
    ax.set_xlabel(col)

axes[0].legend()
plt.tight_layout()
plt.show()

## Saturation Metrics

This section measures whether the trace appears to become committed early.

- irst_low_vocab_entropy_layer: first layer where normalized full-vocabulary entropy falls below the saturation threshold
- 	rue_answer_rank_stable_layer: first layer after which the true answer keeps the same full-vocabulary rank
- 	rue_answer_margin_over_vocab_stable_layer: first layer after which the true answer keeps a non-negative margin against the best competing full-vocabulary token

These are descriptive diagnostics. Some use the true answer and therefore are not deployment-time reliability features, but they are useful for understanding how correct and incorrect trajectories differ.


In [ ]:
saturation_cols = [
    "first_low_vocab_entropy_layer",
    "true_answer_rank_stable_layer",
    "true_answer_margin_over_vocab_stable_layer",
]

display(
    analysis_df.groupby("answer_choice_rank_bucket")[saturation_cols]
    .agg(["mean", "median", "std"])
    .round(4)
)

In [ ]:
saturation_titles = {
    "first_low_vocab_entropy_layer": "First layer with low normalized full-vocabulary entropy",
    "true_answer_rank_stable_layer": "First layer where true-answer full-vocabulary rank becomes stable",
    "true_answer_margin_over_vocab_stable_layer": "First layer where true-answer full-vocabulary margin stays non-negative",
}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

for ax, col in zip(axes, saturation_cols):
    for bucket in bucket_order:
        vals = pd.to_numeric(
            analysis_df.loc[analysis_df["answer_choice_rank_bucket"].eq(bucket), col],
            errors="coerce",
        ).dropna()
        ax.hist(vals, bins=25, density=True, alpha=0.35, color=bucket_colors[bucket], label=bucket)
    ax.set_title(saturation_titles[col])
    ax.set_xlabel("Layer")

axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
layers = np.arange(L_plus_1)

def stack_curve(col):
    return np.stack(analysis_df[col].to_list(), axis=0)


def mean_curve(col, bucket):
    mat = stack_curve(col)
    mask = analysis_df["answer_choice_rank_bucket"].eq(bucket).to_numpy()
    return mat[mask].mean(axis=0)


curve_specs = [
    ("vocab_entropy_normalized_curve", "Normalized full-vocabulary entropy over layers"),
    ("answer_choice_entropy_normalized_curve", "Normalized answer-choice entropy over layers"),
    ("entropy_normalized_difference_curve", "Difference: normalized vocab entropy minus normalized answer-choice entropy"),
    ("true_answer_margin_over_vocab_curve", "True-answer margin over full vocabulary"),
    ("answer_choice_top1_top2_logit_gap_curve", "Answer-choice top1-top2 logit gap over layers"),
    ("true_answer_rank_over_vocab_curve", "True-answer rank over full vocabulary"),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 13))
axes = axes.ravel()

for ax, (col, title) in zip(axes, curve_specs):
    for bucket in bucket_order:
        ax.plot(layers, mean_curve(col, bucket), label=bucket, color=bucket_colors[bucket])
    ax.set_title(title)
    ax.set_xlabel("Layer")

axes[0].legend()
plt.tight_layout()
plt.show()

## Agreement With The Final Prediction

This section asks when intermediate layers already agree with the model's final prediction.

- ull_vocab_prediction_agreement_with_final_curve: whether the full-vocabulary top token at each layer matches the final full-vocabulary top token
- nswer_choice_prediction_agreement_with_final_curve: whether the top answer choice at each layer matches the final answer choice

These metrics do not use the ground-truth answer. They are candidates for later reliability modeling because they only depend on the model's own trace.


In [ ]:
full_vocab_agreement_curve = stack_curve("full_vocab_prediction_agreement_with_final_curve")
answer_choice_agreement_curve = stack_curve("answer_choice_prediction_agreement_with_final_curve")

plt.figure(figsize=(9, 5))
plt.plot(layers, full_vocab_agreement_curve.mean(axis=0), label="full-vocab agreement with final")
plt.plot(layers, answer_choice_agreement_curve.mean(axis=0), label="answer-choice agreement with final")
plt.xlabel("Layer")
plt.ylabel("Dataset fraction")
plt.title("How often each layer already agrees with the final prediction")
plt.legend()
plt.show()


In [ ]:
agreement_specs = [
    ("full_vocab_prediction_agreement_with_final_curve", "Full-vocabulary agreement with final prediction"),
    ("answer_choice_prediction_agreement_with_final_curve", "Answer-choice agreement with final prediction"),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (col, title) in zip(axes, agreement_specs):
    for bucket in bucket_order:
        ax.plot(layers, mean_curve(col, bucket), label=bucket, color=bucket_colors[bucket])
    ax.set_title(title)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Dataset fraction")

axes[0].legend()
plt.tight_layout()
plt.show()


In [ ]:
agreement_summary_cols = [
    "first_full_vocab_agreement_with_final_layer",
    "stable_full_vocab_agreement_with_final_layer",
    "first_answer_choice_agreement_with_final_layer",
    "stable_answer_choice_agreement_with_final_layer",
]

display(
    analysis_df.groupby("answer_choice_rank_bucket")[agreement_summary_cols]
    .agg(["mean", "median", "std"])
    .round(4)
)


In [ ]:
rank_curve = stack_curve("true_answer_rank_within_choices_curve")

hit1 = (rank_curve <= 1).mean(axis=0)
hit2 = (rank_curve <= 2).mean(axis=0)
hit3 = (rank_curve <= 3).mean(axis=0)

plt.figure(figsize=(8, 5))
plt.plot(layers, hit1, label="hit@1")
plt.plot(layers, hit2, label="hit@2")
plt.plot(layers, hit3, label="hit@3")
plt.xlabel("Layer")
plt.ylabel("Dataset fraction")
plt.title("How often the true answer is already in the top-k over layers")
plt.legend()
plt.show()


## Observable Error-Prediction Features

These predictors use only observable quantities from the constrained Aâ€“E decision trace.


In [ ]:
observable_cols = [
    "final_vocab_entropy_normalized",
    "final_answer_choice_entropy_normalized",
    "final_entropy_normalized_difference",
    "final_true_answer_margin_over_vocab",
    "final_answer_choice_top1_top2_logit_gap",
    "answer_choice_winner_flip_count",
    "first_full_vocab_agreement_with_final_layer",
    "stable_full_vocab_agreement_with_final_layer",
    "first_answer_choice_agreement_with_final_layer",
    "stable_answer_choice_agreement_with_final_layer",
]

analysis_df["is_answer_choice_top1_error"] = ~analysis_df["final_answer_choice_is_correct"]
analysis_df["is_answer_choice_rank3plus_hard_miss"] = analysis_df["answer_choice_rank_bucket"].eq("rank3plus_hard")


def build_pipeline():
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED)),
        ]
    )


def single_feature_auc(frame, cols, target):
    rows = []
    y = frame[target].astype(int).to_numpy()
    for col in cols:
        x = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(x)
        if mask.sum() < 30:
            continue
        y_m = y[mask]
        x_m = x[mask]
        if len(np.unique(y_m)) < 2:
            continue
        auc_hi = roc_auc_score(y_m, x_m)
        auc_lo = roc_auc_score(y_m, -x_m)
        if auc_lo > auc_hi:
            auc = auc_lo
            direction = "lower => more risk"
            score = -x_m
        else:
            auc = auc_hi
            direction = "higher => more risk"
            score = x_m
        ap = average_precision_score(y_m, score)
        rows.append({"feature": col, "auroc": auc, "ap": ap, "direction": direction})
    return pd.DataFrame(rows).sort_values(["auroc", "ap"], ascending=False)


print("Single-feature ranking for top-1 error")
display(single_feature_auc(analysis_df, observable_cols, "is_answer_choice_top1_error").round(4))

print("Single-feature ranking for rank>=3 hard misses")
display(single_feature_auc(analysis_df, observable_cols, "is_answer_choice_rank3plus_hard_miss").round(4))


In [ ]:
feature_sets = {
    "entropy_only": ["final_vocab_entropy_normalized", "final_answer_choice_entropy_normalized", "final_entropy_normalized_difference"],
    "vocab_margin_only": ["final_true_answer_margin_over_vocab"],
    "decision_structure_only": [
        "final_answer_choice_top1_top2_logit_gap",
        "answer_choice_winner_flip_count",
        "first_full_vocab_agreement_with_final_layer",
        "stable_full_vocab_agreement_with_final_layer",
        "first_answer_choice_agreement_with_final_layer",
        "stable_answer_choice_agreement_with_final_layer",
    ],
    "entropy_plus_decision_structure": observable_cols,
}


def cross_validated_scores(frame, feature_cols, target):
    X = frame[feature_cols].copy()
    y = frame[target].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    rows = []
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        pipe = build_pipeline()
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        prob = pipe.predict_proba(X.iloc[te_idx])[:, 1]
        rows.append(
            {
                "fold": fold,
                "auroc": roc_auc_score(y[te_idx], prob),
                "ap": average_precision_score(y[te_idx], prob),
            }
        )
    fold_df = pd.DataFrame(rows)
    return {
        "auroc_mean": float(fold_df["auroc"].mean()),
        "auroc_std": float(fold_df["auroc"].std(ddof=1)),
        "ap_mean": float(fold_df["ap"].mean()),
        "ap_std": float(fold_df["ap"].std(ddof=1)),
    }


cv_rows = []
for target in ["is_answer_choice_top1_error", "is_answer_choice_rank3plus_hard_miss"]:
    for name, cols in feature_sets.items():
        scores = cross_validated_scores(analysis_df, cols, target)
        cv_rows.append({"target": target, "model": name, "n_features": len(cols), **scores})

display(pd.DataFrame(cv_rows).sort_values(["target", "auroc_mean"], ascending=[True, False]).round(4))


## Interpretation Notes

For CSQA the most useful non-top1 structure is not a generic full-vocabulary top-k view. It is the **rank of the true answer inside the five answer choices**.

That is why the notebook distinguishes:

- 
ank1_correct
- 
ank2_near_miss
- 
ank3plus_hard

If the layerwise traces show strong separation between those groups, then the hidden-state trajectory contains usable reliability structure even when the final prediction is wrong.
